# Visualize Method Gallery

A complete reference gallery of every supported `visualize()` family
across option models, lattices, SABR smiles, option strategies, portfolio
allocators, and market-data facades. Treat this notebook as a lookup table:
copy the chart name and arguments you need into your own workflow.

**Sections:**
1. Option-model gallery
2. Portfolio gallery
3. Market-data gallery


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
import pandas as pd

from abaquant.derivatives import OptionStrategy
from abaquant.derivatives.models import BlackScholesMertonModel, CoxRossRubinsteinModel
from abaquant.derivatives.models.sabr import SABRVolatilityModel
from abaquant.marketdata import get_ticker, get_tickers
from abaquant.portfolio.optimization import PortfolioAllocator
from abaquant.visualization import VisualizationError

In [ ]:
class DeterministicMarketDataProvider:
    """Minimal offline provider reused across the visualization gallery."""

    name = "deterministic-example"

    def fast_info(self, symbol):
        return {"lastPrice": 105.0}

    def info(self, symbol):
        return {"currency": "USD", "marketCap": 600.0, "symbol": symbol}

    def history(self, symbol, **kwargs):
        import numpy as np
        dates = pd.date_range("2025-01-02", periods=24, freq="B")
        return pd.DataFrame({"Close": 100.0 + np.linspace(0, 8, len(dates))}, index=dates)

    def history_many(self, symbols, **kwargs):
        import numpy as np
        dates = pd.date_range("2025-01-02", periods=24, freq="B")
        data = {s: 100.0 + i * 5 + np.linspace(0, 8, len(dates)) for i, s in enumerate(symbols)}
        return pd.DataFrame(data, index=dates)

    def option_expirations(self, symbol):
        return ["2027-01-15"]

    def option_chain(self, symbol, expiry):
        strikes = [80.0, 90.0, 100.0, 110.0, 120.0]
        calls = pd.DataFrame({
            "contractSymbol": [f"{symbol}C{int(k)}" for k in strikes],
            "strike": strikes, "lastPrice": [22.0, 14.5, 8.0, 4.5, 2.4],
            "bid": [21.5, 14.0, 7.6, 4.1, 2.0], "ask": [22.5, 15.0, 8.4, 4.9, 2.8],
            "impliedVolatility": [0.31, 0.27, 0.23, 0.25, 0.29],
            "openInterest": [120, 240, 520, 310, 180], "volume": [12, 28, 65, 34, 16],
        })
        puts = pd.DataFrame({
            "contractSymbol": [f"{symbol}P{int(k)}" for k in strikes],
            "strike": strikes, "lastPrice": [2.2, 4.1, 7.8, 13.9, 21.0],
            "bid": [1.9, 3.8, 7.4, 13.4, 20.4], "ask": [2.5, 4.4, 8.2, 14.4, 21.6],
            "impliedVolatility": [0.35, 0.30, 0.24, 0.26, 0.32],
            "openInterest": [210, 330, 610, 270, 155], "volume": [18, 36, 70, 29, 14],
        })
        return calls, puts

    def income_statement(self, symbol, *, period="annual"):
        return pd.DataFrame({"2025-12-31": {
            "Total Revenue": 450.0, "EBITDA": 90.0, "EBIT": 75.0,
            "Interest Expense": 10.0, "Net Income": 60.0, "Gross Profit": 200.0,
        }})

    def balance_sheet(self, symbol, *, period="annual"):
        return pd.DataFrame({"2025-12-31": {
            "Total Debt": 120.0, "Stockholders Equity": 300.0, "Current Assets": 250.0,
            "Inventory": 40.0, "Current Liabilities": 100.0, "Cash And Cash Equivalents": 50.0,
            "Total Assets": 500.0, "Total Liabilities": 200.0, "Retained Earnings": 110.0,
            "Long Term Debt": 80.0,
        }})

    def cash_flow_statement(self, symbol, *, period="annual"):
        return pd.DataFrame({"2025-12-31": {"Operating Cash Flow": 70.0}})

## 1. Option-model gallery

Payoff, price profile, extrinsic value, standardized Greeks, price and
gamma surfaces, a binomial lattice, a SABR volatility smile, and option
strategy payoff/component charts.


In [ ]:
def build_option_gallery():
    option = BlackScholesMertonModel(100.0, 105.0, 1.0, 0.05, 0.20)
    lattice = CoxRossRubinsteinModel(100.0, 105.0, 1.0, 0.05, 0.20, number_of_steps=6)
    sabr = SABRVolatilityModel(100.0, 100.0, 1.0, 0.20, 0.5, -0.3, 0.4)
    return {
        "option_payoff": option.visualize(chart="payoff"),
        "option_profile": option.visualize(chart="price_profile"),
        "option_extrinsic": option.visualize(chart="extrinsic_value"),
        "option_greeks": option.visualize(chart="greeks", greek_scale="standardized"),
        "option_price_surface": option.visualize(
            chart="price_surface", grid_size=31, volatility_grid_size=15
        ),
        "option_gamma_surface": option.visualize(
            chart="gamma_surface", grid_size=31, volatility_grid_size=15
        ),
        "lattice_tree": lattice.visualize(chart="tree"),
        "sabr_smile": sabr.visualize(chart="volatility_smile"),
        "strategy_payoff": OptionStrategy.bull_call_spread(
            lower_strike=100.0, upper_strike=115.0, lower_premium=6.0, upper_premium=2.0
        ).visualize(chart="payoff"),
        "strategy_components": OptionStrategy.bull_call_spread(
            lower_strike=100.0, upper_strike=115.0, lower_premium=6.0, upper_premium=2.0
        ).visualize(chart="components"),
    }

## 2. Portfolio gallery

Weights, cumulative returns, and correlation for a maximum-Sharpe allocation.


In [ ]:
def build_portfolio_gallery():
    returns = pd.DataFrame(
        {
            "ALPHA": [0.01, -0.002, 0.006, 0.004, 0.003],
            "BETA": [0.003, 0.005, -0.001, 0.002, 0.004],
            "GAMMA": [-0.002, 0.007, 0.004, 0.006, 0.001],
        }
    )
    allocator = PortfolioAllocator(returns, annual_risk_free_rate=0.02)
    weights = allocator.mean_variance.maximum_sharpe()
    return {
        "portfolio_weights": allocator.visualize(weights=weights, chart="weights"),
        "portfolio_cumulative": allocator.visualize(weights=weights, chart="cumulative_returns"),
        "portfolio_correlation": allocator.visualize(chart="correlation"),
    }

## 3. Market-data gallery

Ticker/universe price history, a financial statement, and option-chain analytics charts.


In [ ]:
def build_market_gallery():
    provider = DeterministicMarketDataProvider()
    ticker = get_ticker("DEMO", provider=provider, financial_cache="memory")
    universe = get_tickers(["ALPHA", "BETA", "GAMMA"], provider=provider)
    assessment = ticker.credit.assess_from_financials()
    chain_analytics = ticker.options.analytics("2027-01-15")
    return {
        "ticker_prices": ticker.visualize(period="1mo"),
        "universe_prices": universe.visualize(period="1mo"),
        "statement": ticker.financials.visualize(statement="balance_sheet"),
        "option_chain_iv_surface": chain_analytics.visualize(chart="iv_surface", option_type="call"),
        "option_chain_rich_cheap": chain_analytics.visualize(
            chart="rich_cheap", option_type="call", risk_free_rate=0.04
        ),
        "credit_metrics": assessment.visualize(chart="metrics"),
        "credit_score": assessment.visualize(chart="score"),
    }

## Build the complete gallery


In [ ]:
try:
    figures = {}
    figures.update(build_option_gallery())
    figures.update(build_portfolio_gallery())
    figures.update(build_market_gallery())
    print(f"Created {len(figures)} figures total:")
    for name, fig in figures.items():
        print(f"  {name:26s}: {type(fig).__name__}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

Use this notebook as a searchable index of chart names. When you know the
domain object (option model, allocator, ticker, ...) but not the exact
`chart=` value, scan the relevant section above.
